# Gemma 4 31B V7.0 Full-Rank Hybrid + CRTR/MoD (`q20`) + Target-Aligned `Assistant-V7` (`BHDR-384`)

**Upgrades over V6.1 (`1.13 bpw`, `2/15`):**
- Full-rank Cholesky Hessian GPTQ + Dual-Ridge ActALS across all 411 LM matrices (`~5.642 GiB <= 5.75 GiB`, **zero 1-bit Attention**, Row-Major `1|0x40` on 153 sliding MLPs with `perm_mlp` co-activation clustering).
- Full 3-bit Vision (`V00..V26`, `~223 MiB`) + Multimodal C8DB (`embed_vision`, `L58 K/V`, `L59 K`, `post_cell60`).
- Real `CRTR` v1 + softer `MoD (q20)` (`10.86 MB`).
- Stage 4B Target-Aligned `Assistant-V7` (`300` steps on `128` V7.0 student sequences) + Q2/Q4/Q8 embed-free bodies + `BHDR d_byte=384` distilled on `V7.0` student greedy argmax (`2400` steps).

In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/DannyB-bit/Language-U-Gemma4.git"
REPO_DIR = Path("/content/Language-U-Gemma4")
DRIVE_V7 = Path("/content/drive/MyDrive/gemma4_v7_0")
DRIVE_V7.mkdir(parents=True, exist_ok=True)

if not REPO_DIR.is_dir():
    subprocess.run(["git", "clone", "-b", "v5-qat-dev", REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", "v5-qat-dev"], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "reset", "--hard", "origin/v5-qat-dev"], check=True)

print(subprocess.check_output(["git", "-C", str(REPO_DIR), "log", "-1", "--oneline"], text=True).strip(), flush=True)
print(subprocess.check_output(["nvidia-smi"], text=True), flush=True)

import torch
assert torch.cuda.is_available(), "CUDA GPU required! Switch Runtime -> Change runtime type -> GPU High-RAM."
vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
smoke_t = (torch.ones((16, 16), device="cuda", dtype=torch.bfloat16) @ torch.ones((16, 16), device="cuda", dtype=torch.bfloat16)).sum().item()
print(f"[GPU Ready] {torch.cuda.get_device_name(0)} | VRAM={vram_gb:.2f} GiB | max_mode={vram_gb >= 64.0} | bf16_smoke={smoke_t:.1f}", flush=True)

_need_pip = False
try:
    import datasets, safetensors, tokenizers, huggingface_hub, PIL
    from transformers.models.gemma4.modeling_gemma4 import Gemma4VisionModel
except Exception:
    _need_pip = True
if _need_pip:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-U",
         "transformers>=4.51.0", "accelerate", "safetensors", "huggingface_hub", "datasets", "tokenizers", "pillow"],
        check=True,
    )

try:
    from google.colab import userdata
    hf_tok = userdata.get("HF_TOKEN")
    if hf_tok:
        os.environ["HF_TOKEN"] = hf_tok
        print("Loaded HF_TOKEN from Colab secrets.")
except Exception:
    pass

SCRIPT = REPO_DIR / "experiments/run270_rust/calib/31b/run_v7_0_fullrank_crtr_gpu.py"
print("Running 6-gate V7.0 CPU pre-flight selftest...", flush=True)
print(subprocess.check_output([sys.executable, str(SCRIPT), "--selftest"], text=True), flush=True)

In [ ]:
import json
import os
import subprocess
import sys
import time
from pathlib import Path

REPO_DIR = Path("/content/Language-U-Gemma4")
subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", "v5-qat-dev"], check=True)
subprocess.run(["git", "-C", str(REPO_DIR), "reset", "--hard", "origin/v5-qat-dev"], check=True)
SCRIPT = REPO_DIR / "experiments/run270_rust/calib/31b/run_v7_0_fullrank_crtr_gpu.py"
LOG_PATH = Path("/content/v7_0_train.log")
DRIVE_V7 = Path("/content/drive/MyDrive/gemma4_v7_0")

import torch
vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
env = os.environ.copy()
env["V7_DRIVE_DIR"] = str(DRIVE_V7)
env["V7_OUT_DIR"] = "/content/v7_0_outputs"
env["V7_MAX_VRAM"] = "1"
env["V7_RESIDENT_GPU"] = "1" if vram_gb >= 64.0 else "0"
env["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True,max_split_size_mb:512"
env["COLAB_AUTO_UNASSIGN"] = "1"
env["PYTHONUNBUFFERED"] = "1"

print(f"Launching V7.0 Full-Rank + CRTR/MoD(q20) + Assistant-V7 BHDR-384 (Max VRAM {vram_gb:.1f} GiB) -> {LOG_PATH}...", flush=True)
with open(LOG_PATH, "w", encoding="utf-8") as lfp:
    proc = subprocess.Popen([sys.executable, "-u", str(SCRIPT), "--run"], stdout=lfp, stderr=subprocess.STDOUT, env=env)

seen = 0
while proc.poll() is None:
    time.sleep(15)
    if LOG_PATH.is_file():
        txt = LOG_PATH.read_text(encoding="utf-8", errors="replace")
        if len(txt) > seen:
            print(txt[seen:], end="", flush=True)
            seen = len(txt)

txt = LOG_PATH.read_text(encoding="utf-8", errors="replace")
if len(txt) > seen:
    print(txt[seen:], end="", flush=True)
assert proc.returncode == 0, f"V7.0 pipeline exited with code {proc.returncode}"

sys.path.insert(0, "/content/Language-U-Gemma4/experiments/run270_rust/calib/31b")
import run_v7_0_fullrank_crtr_gpu as v70
import train_cortical_router as crtr

rpt = json.loads((DRIVE_V7 / "v7_0_report.json").read_text(encoding="utf-8"))
nm, nb = v70.verify_t16c_integrity(DRIVE_V7 / "gemma4_31b_v7_0_fullrank.t16c")
cmeta, clayers = crtr.load_crtr(DRIVE_V7 / "gemma4_31b_v7_router.crtr")
print(f"[VERIFIED 8/8 DRIVE ARTIFACTS] mats={nm} bytes={nb:,} crtr_layers={len(clayers)} flags=0x{cmeta['flags']:x}")
print(json.dumps({k: v for k, v in rpt.items() if k != 'layer_cos_summary'}, indent=2))

try:
    from google.colab import runtime
    print("[Auto-Unassign] Disconnecting Colab runtime...")
    runtime.unassign()
except Exception as exc:
    print(f"[Auto-Unassign Note] {exc}")